In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

# 1. Load ONLY the work dataset
df = pd.read_csv('online_shoppers_WORK.csv')

# 2. Split the WORK data into Train and Validation
# This allows the agent to experiment safely
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['Revenue']
)

# 3. Model Preparation
X_train = train_df.drop(['Revenue', 'Month'], axis=1)
y_train = train_df['Revenue'].astype(int)
X_val = val_df.drop(['Revenue', 'Month'], axis=1)
y_val = val_df['Revenue'].astype(int)

# --- Standard Pipeline Setup ---
categorical_cols = ['VisitorType', 'Weekend', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']
numerical_cols = ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 
                  'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# 4. Train and Evaluate on Validation (Search Phase)
baseline_model.fit(X_train, y_train)
y_val_prob = baseline_model.predict_proba(X_val)[:, 1]

print(f"Validation ROC AUC Score: {roc_auc_score(y_val, y_val_prob):.4f}")

Validation ROC AUC Score: 0.8793
